**Link repo của tôi: https://github.com/ducphat1509-beep/artificial-intelligence**

# Báo cáo Bài tập tuần - Buổi 11: Simulated Annealing và tìm kiếm trong môi trường phức tạp
**Nguyễn Đức Phát - 24110296**

---

## 1. Mục tiêu buổi học
Trong buổi học này, tôi tìm hiểu hai nội dung chính:

1. **Simulated Annealing (Giả kim / Luyện kim mô phỏng)**: một biến thể của Local Search cho phép chấp nhận bước đi xấu hơn theo xác suất.
2. **Tìm kiếm trong môi trường phức tạp**: trường hợp agent không biết đầy đủ trạng thái ban đầu, nên phải tìm kiếm trên **Belief State (BS)**, tức tập các trạng thái có thể xảy ra.

Bài toán minh họa vẫn là 8-Puzzle để giữ sự thống nhất với các buổi trước.

---

## 2. Simulated Annealing
### a. Ý tưởng
Simulated Annealing mô phỏng quá trình luyện kim. Khi nhiệt độ `T` cao, hệ thống còn dao động mạnh nên thuật toán có thể chấp nhận cả bước đi xấu. Khi `T` giảm dần, thuật toán trở nên ổn định hơn và ít chấp nhận bước xấu.

Với bài toán 8-Puzzle, tôi dùng heuristic `h(n)` để đo khoảng cách tới goal. Thuật toán chọn ngẫu nhiên một neighbor rồi tính:

```text
Delta = h(next_state) - h(current_state)
```

Theo slide bài học:
- Nếu `Delta < 0`, trạng thái mới tốt hơn nên nhận ngay.
- Nếu `Delta >= 0`, tính `p = exp(-Delta / T)`.
- Nếu `Random(0, 1) < p`, vẫn nhận trạng thái mới.
- Sau mỗi vòng, giảm nhiệt độ: `T = alpha * T`.

### b. Lưu ý về T và alpha
Nhiệt độ cao không đồng nghĩa với tỉ lệ thành công cao. Nếu `T = 10` và `alpha` nằm trong khoảng `0.95 - 0.99`, thuật toán có thể nhận bước xấu quá thường xuyên. Ví dụ `Delta = 2` thì:

```text
exp(-2 / 10) ≈ 0.819
```

Tức là bước xấu vẫn được chấp nhận với xác suất hơn 80%. Khi `alpha` quá gần 1, nhiệt độ giảm chậm, thuật toán dễ lang thang giống random walk. Vì vậy trong chương trình tôi để mặc định `T0 = 2.0`, `alpha = 0.94`, nhưng vẫn cho phép thay đổi để quan sát.

### c. Bốn đặc trưng
- **Tính đầy đủ**: Không đảm bảo trong phiên bản hữu hạn.
- **Tính tối ưu**: Không đảm bảo tìm đường đi ngắn nhất.
- **Độ phức tạp thời gian**: Xấp xỉ `O(t * b * cost(h))`.
- **Độ phức tạp không gian**: Thấp, thường `O(1)` hoặc `O(b)` nếu cần hiển thị neighbor.

---

## 3. Tìm kiếm trong môi trường phức tạp và Belief State
### a. Ý tưởng
Ở các bài trước, agent biết chính xác initial state. Trong môi trường phức tạp, agent có thể chỉ biết một phần thông tin. Khi đó, thay vì một trạng thái ban đầu duy nhất, agent có một tập trạng thái có thể xảy ra, gọi là **Belief State**.

Trong ví dụ 8-Puzzle:
- Agent biết vị trí của 3 ô số: `2`, `8`, `3`.
- Agent biết vị trí ô trống `0`.
- Còn 5 ô số khác chưa biết.

Do đó số trạng thái có thể là:

```text
5! = 120
```

### b. Cập nhật Belief State
Một bước tìm kiếm trên Belief State gồm:

1. Chọn một action hợp lệ với toàn bộ các trạng thái trong BS.
2. Áp dụng action lên từng trạng thái trong BS.
3. Nhận quan sát mới từ môi trường.
4. Lọc bỏ các trạng thái không phù hợp với quan sát.

Trong minh họa này, môi trường thật được ẩn bên trong chương trình. Sau mỗi action, agent quan sát được **ô vừa trượt vào vị trí blank**, nhờ đó biết thêm một tile mới. Vì vậy BS có thể giảm từ:

```text
5! = 120  ->  4! = 24  ->  3! = 6  -> ...
```

### c. Bốn đặc trưng
- **Tính đầy đủ**: Phụ thuộc thuật toán chọn action trên BS; demo tham lam không đảm bảo đầy đủ.
- **Tính tối ưu**: Không đảm bảo tối ưu.
- **Độ phức tạp thời gian**: Cao hơn tìm kiếm thường, khoảng `O(|BS| * b * cost(h))` cho mỗi bước.
- **Độ phức tạp không gian**: `O(|BS|)` vì phải lưu tập trạng thái có thể.

---

## 4. Hướng dẫn chạy
1. Chạy cell code bên dưới để mở giao diện Tkinter.
2. Chọn `Simulated Annealing` để quan sát công thức `Delta`, `p`, `roll` và nhiệt độ `T`.
3. Chọn `Belief State Demo` để quan sát `BS = 5! = 120` rồi giảm dần khi có quan sát mới.
4. Nhấn `Reset`, sau đó dùng `Next Step` hoặc `Auto Run`.


In [ ]:
# =========================================================================
# BUỔI 11 - SIMULATED ANNEALING & BELIEF STATE DEMO TRÊN 8-PUZZLE
# - Code cell viết lại gọn hơn để chạy ổn định trong notebook.
# - Giữ nguyên PUZZLE_START và PUZZLE_GOAL theo case study hiện tại.
# =========================================================================

import itertools
import math
import random
import tkinter as tk
from tkinter import ttk

PUZZLE_START = ((2, 8, 3), (1, 6, 4), (7, 0, 5))
PUZZLE_GOAL = ((1, 2, 3), (8, 0, 4), (7, 6, 5))
MOVES = [
    (-1, 0, 'Up'),
    (1, 0, 'Down'),
    (0, -1, 'Left'),
    (0, 1, 'Right'),
]

DEFAULT_T0 = 2.0
DEFAULT_ALPHA = 0.94
DEFAULT_TMIN = 0.001
DEFAULT_MAX_STEPS = 500
OBSERVED_TILES = (2, 8, 3, 0)


# ── 8-Puzzle helpers ─────────────────────────────────────
def find_blank(state):
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:
                return r, c
    raise ValueError('State không có ô trống 0')


def get_neighbors(state):
    x, y = find_blank(state)
    result = []
    for dx, dy, action in MOVES:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_state = [list(row) for row in state]
            new_state[x][y], new_state[nx][ny] = new_state[nx][ny], new_state[x][y]
            result.append((tuple(tuple(row) for row in new_state), action))
    return result


def apply_action(state, action_name):
    for neighbor, action in get_neighbors(state):
        if action == action_name:
            return neighbor
    return None


def moved_tile_for_action(state, action_name):
    x, y = find_blank(state)
    for dx, dy, action in MOVES:
        if action == action_name:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                return state[nx][ny]
    return None


def manhattan_distance(state, goal=PUZZLE_GOAL):
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    total = 0
    for r in range(3):
        for c in range(3):
            value = state[r][c]
            if value != 0:
                gr, gc = goal_pos[value]
                total += abs(r - gr) + abs(c - gc)
    return total


def misplaced_tiles(state, goal=PUZZLE_GOAL):
    return sum(
        1
        for r in range(3)
        for c in range(3)
        if state[r][c] != 0 and state[r][c] != goal[r][c]
    )


def choose_heuristic(name):
    return misplaced_tiles if name == 'Misplaced Tiles' else manhattan_distance


def is_goal(state):
    return state == PUZZLE_GOAL


def make_node(state, heuristic, action='Current', status='', note=''):
    return {
        'state': state,
        'h': heuristic(state) if isinstance(state[0][0], int) else '?',
        'action': action,
        'status': status,
        'note': note,
    }


def positions_of(state):
    result = {}
    for r in range(3):
        for c in range(3):
            result[state[r][c]] = (r, c)
    return result


# ── Simulated Annealing ──────────────────────────────────
def simulated_annealing_gen(start, heuristic, t0=DEFAULT_T0, alpha=DEFAULT_ALPHA, tmin=DEFAULT_TMIN, max_steps=DEFAULT_MAX_STEPS):
    current = start
    current_h = heuristic(current)
    temperature = t0
    expanded = 0
    generated = 0
    explored = []

    for step in range(max_steps + 1):
        explored.append(current)
        if is_goal(current):
            yield make_node(current, heuristic, note=f'T={temperature:.4f}'), [], explored, expanded, generated, [current], 'Simulated Annealing - GOAL'
            return
        if temperature <= tmin:
            yield make_node(current, heuristic, note=f'T={temperature:.4f}'), [], explored, expanded, generated, None, 'Simulated Annealing STOP - T <= Tmin'
            return

        neighbors = get_neighbors(current)
        generated += len(neighbors)
        next_state, action = random.choice(neighbors)
        next_h = heuristic(next_state)
        delta = next_h - current_h

        if delta < 0:
            probability = 1.0
            roll = None
            accepted = True
            decision = 'Delta < 0, accept directly'
        else:
            probability = math.exp(-delta / temperature)
            roll = random.random()
            accepted = roll < probability
            decision = f'Delta >= 0, p={probability:.3f}, roll={roll:.3f}'

        frontier = []
        for state, move in neighbors:
            item = make_node(state, heuristic, action=move, status='Rejected')
            if state == next_state:
                item['status'] = 'Accepted' if accepted else 'Sampled-Rejected'
                item['note'] = f'Delta={delta}, p={probability:.3f}, roll={"direct" if roll is None else f"{roll:.3f}"}'
            elif heuristic(state) < current_h:
                item['status'] = 'Better'
            frontier.append(item)

        mode = f'SA | T={temperature:.4f} | sample={action} | Delta={delta} | {decision}'
        yield make_node(current, heuristic, note=f'h={current_h}, step={step}'), frontier, explored, expanded, generated, None, mode

        if accepted:
            current = next_state
            current_h = next_h
            expanded += 1
        temperature *= alpha

    yield make_node(current, heuristic), [], explored, expanded, generated, None, 'Simulated Annealing STOP - max_steps'


# ── Belief State helpers ─────────────────────────────────
def observe_tiles(state, tiles):
    pos = positions_of(state)
    return {tile: pos[tile] for tile in tiles}


def pattern_from_known(known):
    grid = [['?' for _ in range(3)] for _ in range(3)]
    for tile, (r, c) in known.items():
        grid[r][c] = tile
    return tuple(tuple(row) for row in grid)


def build_state_from_assignment(assignment):
    grid = [[None for _ in range(3)] for _ in range(3)]
    for tile, (r, c) in assignment.items():
        grid[r][c] = tile
    return tuple(tuple(row) for row in grid)


def belief_from_known(known):
    used_tiles = set(known)
    used_cells = set(known.values())
    remaining_tiles = sorted(set(range(9)) - used_tiles)
    remaining_cells = [(r, c) for r in range(3) for c in range(3) if (r, c) not in used_cells]

    belief = []
    for perm in itertools.permutations(remaining_tiles):
        assignment = dict(known)
        for tile, cell in zip(perm, remaining_cells):
            assignment[tile] = cell
        belief.append(build_state_from_assignment(assignment))
    return belief


def filter_belief(states, known):
    result = []
    for state in states:
        pos = positions_of(state)
        if all(pos[tile] == cell for tile, cell in known.items()):
            result.append(state)
    return result


def legal_actions_for_all(states):
    common = None
    for state in states:
        actions = {action for _, action in get_neighbors(state)}
        common = actions if common is None else common & actions
    return [action for _, _, action in MOVES if action in common]


def transition_belief(states, action):
    return [apply_action(state, action) for state in states if apply_action(state, action) is not None]


def belief_stats(states, heuristic):
    values = [heuristic(state) for state in states]
    return min(values), sum(values) / len(values), max(values)


def best_representative(states, heuristic):
    return min(states, key=lambda state: (heuristic(state), state))


def diverse_samples(states, heuristic, limit=3):
    ordered = sorted(states, key=lambda state: (heuristic(state), state))
    if len(ordered) <= limit:
        return ordered
    indexes = [0, len(ordered) // 2, len(ordered) - 1]
    samples = []
    seen = set()
    for idx in indexes:
        state = ordered[idx]
        if state not in seen:
            samples.append(state)
            seen.add(state)
    return samples


def belief_state_gen(heuristic, max_steps=20):
    true_state = PUZZLE_START
    known = observe_tiles(true_state, OBSERVED_TILES)
    belief = belief_from_known(known)
    expanded = 0
    generated = 0
    explored = []

    for step in range(max_steps + 1):
        explored.append(tuple(belief[:5]))
        bmin, bavg, bmax = belief_stats(belief, heuristic)
        pattern = pattern_from_known(known)
        current = [
            {
                'state': pattern,
                'h': '?',
                'action': 'Known / Unknown',
                'status': 'Pattern',
                'note': f'known={sorted(known)} | BS={len(belief)} | min={bmin}, avg={bavg:.2f}, max={bmax}',
            }
        ]
        for i, sample in enumerate(diverse_samples(belief, heuristic), start=1):
            current.append(make_node(sample, heuristic, action=f'Possible state {i}', status='Sample'))

        if is_goal(true_state):
            yield current, [], explored, expanded, generated, [true_state], 'Belief State - TRUE STATE reached GOAL'
            return

        actions = legal_actions_for_all(belief)
        true_legal = {action for _, action in get_neighbors(true_state)}
        actions = [action for action in actions if action in true_legal]
        if not actions:
            yield current, [], explored, expanded, generated, None, 'Belief State STOP - no common legal action'
            return

        frontier = []
        for action in actions:
            moved_tile = moved_tile_for_action(true_state, action)
            next_true = apply_action(true_state, action)
            next_known = dict(known)
            next_known.update(observe_tiles(next_true, (0, moved_tile)))

            transitioned = transition_belief(belief, action)
            filtered = filter_belief(transitioned, next_known)
            generated += len(transitioned)
            if not filtered:
                continue

            fmin, favg, fmax = belief_stats(filtered, heuristic)
            rep = best_representative(filtered, heuristic)
            item = make_node(rep, heuristic, action=action, status='Rejected')
            item['belief_before'] = len(transitioned)
            item['belief_after'] = len(filtered)
            item['belief_min'] = fmin
            item['belief_avg'] = favg
            item['belief_max'] = fmax
            item['next_true'] = next_true
            item['next_known'] = next_known
            item['next_belief'] = filtered
            item['note'] = f'reveal tile {moved_tile}: BS {len(transitioned)} -> {len(filtered)}'
            frontier.append(item)

        if not frontier:
            yield current, [], explored, expanded, generated, None, 'Belief State STOP - observation removed all states'
            return

        frontier.sort(key=lambda item: (item['belief_avg'], item['belief_min'], item['belief_after'], item['action']))
        chosen = frontier[0]
        chosen['status'] = 'Chosen'
        mode = f"Belief State | choose={chosen['action']} | {chosen['note']}"
        yield current, frontier, explored, expanded, generated, None, mode

        true_state = chosen['next_true']
        known = chosen['next_known']
        belief = chosen['next_belief']
        expanded += 1

    yield current, [], explored, expanded, generated, None, 'Belief State STOP - max_steps'


# ── GUI ──────────────────────────────────────────────────
class ScrollableFrame(tk.Frame):
    def __init__(self, parent, height=280):
        super().__init__(parent)
        self.canvas = tk.Canvas(self, height=height, highlightthickness=0)
        self.scroll_x = tk.Scrollbar(self, orient='horizontal', command=self.canvas.xview)
        self.inner = tk.Frame(self.canvas)
        self.inner.bind('<Configure>', lambda event: self.canvas.configure(scrollregion=self.canvas.bbox('all')))
        self.canvas.create_window((0, 0), window=self.inner, anchor='nw')
        self.canvas.configure(xscrollcommand=self.scroll_x.set)
        self.canvas.pack(side='top', fill='both', expand=True)
        self.scroll_x.pack(side='bottom', fill='x')


class PuzzleGUI:
    def __init__(self, root):
        self.root = root
        self.gen = None
        self.running = False
        self.step_count = 0

        root.title('Buổi 11 - Simulated Annealing & Belief State')
        sw, sh = root.winfo_screenwidth(), root.winfo_screenheight()
        w, h = min(1400, int(sw * .85)), min(900, int(sh * .85))
        root.geometry(f'{w}x{h}+{(sw-w)//2}+{(sh-h)//2}')
        root.minsize(940, 620)

        self.info = tk.Label(root, text='Buổi 11 - chọn thuật toán rồi Reset', font=('Arial', 14, 'bold'))
        self.info.pack(pady=6)

        self.cur_frame = tk.LabelFrame(root, text='Current State / Belief State', padx=8, pady=8)
        self.cur_frame.pack(side='top', fill='x', padx=10, pady=4)

        self.fl_frame = tk.LabelFrame(root, text='Frontier / Candidates', padx=8, pady=8)
        self.fl_frame.pack(side='top', fill='both', expand=True, padx=10, pady=4)
        self.frontier_sf = ScrollableFrame(self.fl_frame)
        self.frontier_sf.pack(fill='both', expand=True)

        bottom = tk.Frame(root)
        bottom.pack(side='bottom', fill='x', pady=8)
        center = tk.Frame(bottom)
        center.pack(anchor='center')

        tk.Label(center, text='Algorithm:', font=('Arial', 10, 'bold')).pack(side='left', padx=(5, 2))
        self.algo_cb = ttk.Combobox(center, values=['Simulated Annealing', 'Belief State Demo'], state='readonly', width=24)
        self.algo_cb.set('Simulated Annealing')
        self.algo_cb.pack(side='left', padx=5)

        tk.Label(center, text='Heuristic:', font=('Arial', 10, 'bold')).pack(side='left', padx=(10, 2))
        self.heuristic_cb = ttk.Combobox(center, values=['Manhattan Distance', 'Misplaced Tiles'], state='readonly', width=18)
        self.heuristic_cb.set('Manhattan Distance')
        self.heuristic_cb.pack(side='left', padx=5)

        tk.Label(center, text='T0').pack(side='left', padx=(8, 2))
        self.t0_var = tk.DoubleVar(value=DEFAULT_T0)
        tk.Spinbox(center, from_=0.1, to=50.0, increment=0.1, textvariable=self.t0_var, width=6).pack(side='left')

        tk.Label(center, text='alpha').pack(side='left', padx=(8, 2))
        self.alpha_var = tk.DoubleVar(value=DEFAULT_ALPHA)
        tk.Spinbox(center, from_=0.80, to=0.999, increment=0.01, textvariable=self.alpha_var, width=6).pack(side='left')

        tk.Label(center, text='steps').pack(side='left', padx=(8, 2))
        self.steps_var = tk.IntVar(value=DEFAULT_MAX_STEPS)
        tk.Spinbox(center, from_=50, to=5000, increment=50, textvariable=self.steps_var, width=6).pack(side='left')

        tk.Button(center, text='Reset', width=8, command=self.reset_search, bg='#FF5722', fg='white').pack(side='left', padx=(10, 5))
        tk.Button(center, text='Next Step', width=12, command=self.next_step).pack(side='left', padx=5)
        self.auto_btn = tk.Button(center, text='Auto Run', width=12, command=self.toggle_auto)
        self.auto_btn.pack(side='left', padx=5)

        self.speed_var = tk.IntVar(value=320)
        tk.Label(center, text='Speed:').pack(side='left', padx=(12, 2))
        tk.Scale(center, from_=10, to=1000, orient='horizontal', variable=self.speed_var, showvalue=False, length=120).pack(side='left')

        self.reset_search()

    def current_heuristic(self):
        return choose_heuristic(self.heuristic_cb.get())

    def reset_search(self):
        self.running = False
        self.step_count = 0
        self.auto_btn.config(text='Auto Run')
        self.clear(self.cur_frame)
        self.clear(self.frontier_sf.inner)

        heuristic = self.current_heuristic()
        if self.algo_cb.get() == 'Belief State Demo':
            self.gen = belief_state_gen(heuristic)
            known = observe_tiles(PUZZLE_START, OBSERVED_TILES)
            belief = belief_from_known(known)
            self.draw_state(self.cur_frame, pattern_from_known(known), f'Initial partial observation\nBS = 5! = {len(belief)}').pack(side='left', padx=8)
            self.fl_frame.config(text='Belief actions: transition -> observe moved tile -> filter BS')
        else:
            self.gen = simulated_annealing_gen(
                PUZZLE_START,
                heuristic,
                t0=float(self.t0_var.get()),
                alpha=float(self.alpha_var.get()),
                max_steps=int(self.steps_var.get()),
            )
            self.draw_state(self.cur_frame, PUZZLE_START, f'Initial Start\nh={heuristic(PUZZLE_START)}').pack(side='left', padx=8)
            self.fl_frame.config(text='Sampled neighbor and other neighbors')

        self.info.config(text='Ready! Click Next Step or Auto Run')

    def draw_state(self, parent, state, label_text='', bg_color='white'):
        container = tk.Frame(parent, bg=bg_color, bd=2, relief='groove')
        grid_frame = tk.Frame(container, bg=bg_color)
        grid_frame.pack(padx=4, pady=4)
        for r in range(3):
            for c in range(3):
                value = state[r][c]
                if value == '?':
                    text = '?'
                    cell_bg = '#FFF59D'
                else:
                    text = str(value) if value else ''
                    cell_bg = '#4CAF50' if value == 0 else 'white'
                tk.Label(grid_frame, text=text, width=4, height=2, font=('Arial', 13, 'bold'), relief='solid', bg=cell_bg).grid(row=r, column=c)
        if label_text:
            tk.Label(container, text=label_text, font=('Arial', 9, 'bold'), fg='black', bg=bg_color, justify='center').pack(pady=4)
        return container

    def clear(self, frame):
        for widget in frame.winfo_children():
            widget.destroy()

    def render(self, current, frontier, explored, expanded, generated, path, mode):
        self.clear(self.cur_frame)
        self.clear(self.frontier_sf.inner)

        for item in current[:4]:
            label = f"{item['action']}\nh={item['h']}"
            if item.get('note'):
                label += f"\n{item['note']}"
            self.draw_state(self.cur_frame, item['state'], label).pack(side='left', padx=8)

        for index, item in enumerate(frontier):
            label = f"{item['action']}\nh={item['h']}"
            if 'belief_after' in item:
                label += f"\nBS {item['belief_before']} -> {item['belief_after']}"
                label += f"\nmin={item['belief_min']}, avg={item['belief_avg']:.2f}, max={item['belief_max']}"
            if item.get('note'):
                label += f"\n{item['note']}"

            bg = 'white'
            if item['status'] in ('Accepted', 'Chosen'):
                bg = '#C8E6C9'
                label += '\n[CHOSEN]' if item['status'] == 'Chosen' else '\n[ACCEPTED]'
            elif item['status'] == 'Sampled-Rejected':
                bg = '#FFCDD2'
                label += '\n[SAMPLED - REJECTED]'
            elif item['status'] == 'Better':
                bg = '#BBDEFB'
                label += '\n[Better]'

            self.draw_state(self.frontier_sf.inner, item['state'], label, bg).grid(row=0, column=index, padx=8, pady=8)

        text = f'[{mode}] Step: {self.step_count} | Expanded: {expanded} | Generated: {generated} | Snapshots: {len(explored)}'
        if path:
            text += ' | GOAL'
        if 'STOP' in mode:
            text += ' | STOPPED'
        self.info.config(text=text)

    def next_step(self):
        try:
            current, frontier, explored, expanded, generated, path, mode = next(self.gen)
            self.step_count += 1
            current_items = current if isinstance(current, list) else [current]
            self.render(current_items, frontier, explored, expanded, generated, path, mode)
            if path or 'STOP' in mode:
                self.running = False
                self.auto_btn.config(text='Auto Run')
        except StopIteration:
            self.running = False
            self.auto_btn.config(text='Done')

    def toggle_auto(self):
        self.running = not self.running
        self.auto_btn.config(text='Pause' if self.running else 'Auto Run')
        if self.running:
            self.auto_tick()

    def auto_tick(self):
        if not self.running:
            return
        self.next_step()
        if self.running:
            self.root.after(self.speed_var.get(), self.auto_tick)


root = tk.Tk()
app = PuzzleGUI(root)
root.mainloop()
